# ReviewIQ — Baseline Comparisons

No model choice is credible without a baseline. This notebook reproduces the two
comparisons quoted in `docs/PRESENTATION_QA.md` §3, on the Netflix 10k prototype:

1. **Sentiment:** VADER (rule-based, no ML) vs our multilingual transformer
2. **Topic discovery:** KMeans (fixed k, no noise concept) vs UMAP + HDBSCAN

Inputs: `data/processed/embeddings/netflix_clustered.parquet` (reviews + transformer
sentiment) and `netflix_sample.npy` (embeddings). Run `04.pipeline.ipynb` first if missing.

## 1. Sentiment: VADER baseline vs transformer

In [1]:
import pandas as pd
import nltk
nltk.download("vader_lexicon", quiet=True)   # tiny word-list file, cached after first run
from nltk.sentiment import SentimentIntensityAnalyzer

reviews = pd.read_parquet("../data/processed/embeddings/netflix_clustered.parquet")

# Benchmark set: 150 clearly-angry (1-star) + 150 clearly-happy (5-star) reviews.
# Star ratings serve as ground truth the authors gave us themselves.
neg = reviews[reviews["score"] == 1].sample(150, random_state=42)
pos = reviews[reviews["score"] == 5].sample(150, random_state=42)
bench = pd.concat([neg, pos])

sia = SentimentIntensityAnalyzer()

def vader_label(text):
    # VADER outputs a "compound" score from -1 to +1; standard thresholds:
    c = sia.polarity_scores(text)["compound"]
    if c >= 0.05:  return "positive"
    if c <= -0.05: return "negative"
    return "neutral"

bench["vader"] = bench["content"].map(vader_label)
print("VADER: 1-star called negative:", f'{(bench[bench["score"]==1]["vader"]=="negative").mean():.0%}')
print("VADER: 5-star called positive:", f'{(bench[bench["score"]==5]["vader"]=="positive").mean():.0%}')

VADER: 1-star called negative: 51%
VADER: 5-star called positive: 88%


In [2]:
# Our transformer's labels are already stored in the parquet — same rows, same benchmark:
print("Transformer: 1-star called negative:",
      f'{(bench[bench["score"]==1]["sentiment"]=="negative").mean():.0%}')
print("Transformer: 5-star called positive:",
      f'{(bench[bench["score"]==5]["sentiment"]=="positive").mean():.0%}')

Transformer: 1-star called negative: 73%
Transformer: 5-star called positive: 87%


## 2. Topic discovery: KMeans baseline vs UMAP + HDBSCAN

In [3]:
import numpy as np, umap
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

embeddings = np.load("../data/processed/embeddings/netflix_sample.npy")

# Baseline A: KMeans straight on the raw 384-dimensional vectors.
# (KMeans must be TOLD the cluster count; we give it 31 — which we only know
# because HDBSCAN discovered it. The baseline gets a head start and still loses.)
km_raw = KMeans(n_clusters=31, random_state=42, n_init=10).fit_predict(embeddings)
print("KMeans on raw 384-dim :", f"{silhouette_score(embeddings, km_raw, sample_size=5000, random_state=42):.3f}")

# Baseline B: same KMeans after UMAP squashes 384 -> 5 dimensions
emb5 = umap.UMAP(n_neighbors=15, n_components=5, metric="cosine",
                 random_state=42, init="random").fit_transform(embeddings)
km_5d = KMeans(n_clusters=31, random_state=42, n_init=10).fit_predict(emb5)
print("KMeans on UMAP 5-dim  :", f"{silhouette_score(emb5, km_5d, sample_size=5000, random_state=42):.3f}")

KMeans on raw 384-dim : 0.035


C:\Users\nithi\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


KMeans on UMAP 5-dim  : 0.341


In [4]:
import hdbscan

hd = hdbscan.HDBSCAN(min_cluster_size=30, min_samples=5).fit_predict(emb5)
mask = hd != -1   # silhouette is computed on clustered points; noise is excluded (stated openly)

print("HDBSCAN clusters:", len(set(hd)) - 1, "| noise:", f"{(~mask).mean():.0%}")
print("HDBSCAN silhouette (non-noise):",
      f"{silhouette_score(emb5[mask], hd[mask], sample_size=5000, random_state=42):.3f}")

# Verdict: silhouette is a TIE with KMeans on the same reduced space (~0.34) —
# the honest claim is not "higher score" but: HDBSCAN discovers the topic count
# itself, and quarantines ~26% generic filler ("good app") as noise instead of
# smearing it into every topic like KMeans does.

HDBSCAN clusters: 31 | noise: 26%


HDBSCAN silhouette (non-noise): 0.342
